# Рекомендательная система для участия поставщиков в госзакупках

Цель: разработать рекомендательную систему, которая поможет оценить поставщику целесообразность своего участия в госзакупках.

Задачи:
1) Проверить датасет на аномилии (дубликаты, пропуски)
2) Привести данные в удобный для анализа вид (парсинг, склеивание строк)
3) Группировка лотов по поставщикам (создание профилей поставщиков)
4) Создание решающего правила (максимум суммы схожести параметров поставщика с параметрами профилей поставщиков)
5) Выдача топ-10 наиболее релеватных лотов.
6) Сравнение того, в чем данный поставщик уже принимал участие (проверка адеватности рекомендаций).

Структура датасета:
- Fz – федеральный закон, по которому проходит закупка
- pn_lot – как раз ключ (по сути это склейка номера закупки и лота
- region_code – регион поставки
- min_publish_date – дата публикации лота
- purchase_name – наименование закупки
- lot_name – наименование лота
- forsmallbiz – закупка относится в малому и среднему бизнесу
- lot_price – НМЦ лота
- customer_inn_kpp – ИНН и КПП заказчика
- item_descriptions – потоварка в текстовом виде в лоте
- okpd2_codes – коды окпд2 лота (может быть несколько, по сути это категории лота)
- participant_inn_kpp – обезличенный участник  
- is_winner – 0 только участие, без победы, 1 – победа.

## 1. Предобработка данных

In [4]:
import pandas as pd
import numpy as np
import re

In [5]:
df = pd.read_excel('Dannnye_dlya_razrabotki_rek_servisa_20261120.xlsx')

In [6]:
print(df.shape)
display(df.head())

(861748, 13)


,fz,pn_lot,region_code,min_publish_date,purchase_name,lot_name,forsmallbiz,lot_price,customer_inn_kpp,item_descriptions,okpd2_codes,participant_inn_kpp,is_winner
0,44fz,0345200019825000037_,47,2025-04-10,Поставка фотобарабанов (совместимых) для много...,NaN,True,3000000.0,7842453565_470301001,Фотобарабан 40IU40R || Фотобарабан 60IU60S,26.20.40.120,F04E2C1D20D9B61B7C99B48EBAE5C0D7,1
1,44fz,0345200019825000038_,47,2025-04-10,Поставка фотобарабанов и блоков проявки (ориги...,NaN,True,3000000.0,7842453565_470301001,Блок проявки DV-1150 || Фотобарабан DK-1150 ||...,26.20.40.120,9F89EA463592C89553736DDFB9CBF73C,1
2,44fz,0345200019825000046_,47,2025-05-05,Поставка IP-видеорегистраторов и камер видеона...,NaN,True,1652750.0,7842453565_470301001,Камера видеонаблюдения,26.70.13.000 | 26.40.33.190,CB37D3E419F63C7D0833A6C6AE828793,1
3,44fz,0345200019825000050_,47,2025-05-07,Поставка штемпельной продукции для филиала «Му...,NaN,True,141150.0,7842453565_470301001,Штемпель,NaN,58D8A3BA3536D51D39525FD1F8E8EEFC,1
4,44fz,0345200019825000051_,47,2025-05-07,Поставка маршрутизаторов,NaN,True,1005000.0,7842453565_470301001,Маршрутизатор,NaN,1C8A8B3933EE4C539BC98F9EEC0C3C49,1


In [7]:
print("\n=== ОБЩАЯ СВОДКА ===")
print("Строк:", len(df))
print("Поставщиков:", df["participant_inn_kpp"].nunique())
print("Лотов:", df["pn_lot"].nunique())

print("\nТипы данных:")
print(df.dtypes)


=== ОБЩАЯ СВОДКА ===
Строк: 861748
Поставщиков: 111448
Лотов: 816803

Типы данных:
fz                      object
pn_lot                  object
region_code              int64
min_publish_date        object
purchase_name           object
lot_name                object
forsmallbiz               bool
lot_price              float64
customer_inn_kpp        object
item_descriptions       object
okpd2_codes             object
participant_inn_kpp     object
is_winner                int64
dtype: object


In [8]:
print("\n=== ДУБЛИКАТЫ ===")
# исключение производных колонок со списками/текстом
raw_cols = [c for c in df.columns if c not in ("okpd_list", "text")]

full_dup = df[raw_cols].duplicated().sum()
print(f"Полных дубликатов строк: {full_dup}")

key_dup = df.duplicated(subset=["participant_inn_kpp", "pn_lot"]).sum()
print(f"Дубликатов по (поставщик + лот): {key_dup}")


=== ДУБЛИКАТЫ ===
Полных дубликатов строк: 0
Дубликатов по (поставщик + лот): 0


In [9]:
print("\n=== ПУСТЫЕ ЗНАЧЕНИЯ ===")
na_counts = df.isna().sum()
na_pct = (df.isna().mean() * 100).round(1)
na_table = pd.DataFrame({"пусто, шт": na_counts, "пусто, %": na_pct})
na_table = na_table[na_table["пусто, шт"] > 0].sort_values("пусто, %", ascending=False)
print(na_table if len(na_table) else "Пустых значений нет")


=== ПУСТЫЕ ЗНАЧЕНИЯ ===
                   пусто, шт  пусто, %
lot_name              784230      91.0
okpd2_codes           400871      46.5
purchase_name          81601       9.5
item_descriptions      54490       6.3


In [10]:
print("\n=== УНИКАЛЬНЫЕ ЗНАЧЕНИЯ В КЛЮЧЕВЫХ КОЛОНКАХ ===")
for col in ["fz", "forsmallbiz", "is_winner"]:
    if col in df.columns:
        print(f"{col}: {df[col].value_counts().to_dict()}")


=== УНИКАЛЬНЫЕ ЗНАЧЕНИЯ В КЛЮЧЕВЫХ КОЛОНКАХ ===
fz: {'44fz': 784230, '223fz': 77518}
forsmallbiz: {True: 520870, False: 340878}
is_winner: {1: 784920, 0: 76828}


In [11]:
print("\n=== РЕГИОНЫ ===")
print(f"Уникальных регионов: {df['region_code'].nunique()}")
print(f"Топ-5 регионов: {df['region_code'].value_counts().head(5).to_dict()}")


=== РЕГИОНЫ ===
Уникальных регионов: 95
Топ-5 регионов: {78: 57630, 50: 52307, 54: 51713, 23: 38062, 74: 32073}


In [12]:
print("\n=== ЦЕНЫ ===")
print(df["lot_price"].describe().round(0))


=== ЦЕНЫ ===
count    8.617480e+05
mean     4.974384e+06
std      1.089163e+08
min      0.000000e+00
25%      1.168200e+05
50%      4.380000e+05
75%      1.494000e+06
max      4.796740e+10
Name: lot_price, dtype: float64


In [13]:
print("\n=== УЧАСТИЙ НА ПОСТАВЩИКА ===")
print(df.groupby("participant_inn_kpp").size().describe())


=== УЧАСТИЙ НА ПОСТАВЩИКА ===
count    111448.000000
mean          7.732288
std          31.863401
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max        3002.000000
dtype: float64


## 2. Выделение категорий

ОКПД2 - главный сигнал категории, но в ячейке он лежит строкой с разделителем (либо |, либо точка с запятой, либо запятая) или пустой. 
Текст лота размазан по трем колонкам (purchase_name, lot_name, item_descriptions), причём lot_name часто пустой - склеиваем все в одну строку, чтобы потом искать пересечения слов. 

In [16]:
def parse_okpd(x):
    """'Парсинг ОКПД2.
    Берет ячейку с кодами ОКПД2, проверяет что там не пусто,
    разбивает строку по разделителям,
    очищает каждый кусок от пробелов и выбрасывает пустые,
    возвращает чистый список кодов.
    Если ячейка пустая, то возвращает пустой список.
    Результат в колонку okpd_list.
    """
    
    if pd.isna(x) or str(x).strip() == "":
        return []
    parts = re.split(r"[|;,]", str(x))
    return [part.strip() for part in parts if part.strip()]

df["okpd_list"] = df["okpd2_codes"].apply(parse_okpd)

In [17]:
display(df.head())

,fz,pn_lot,region_code,min_publish_date,purchase_name,lot_name,forsmallbiz,lot_price,customer_inn_kpp,item_descriptions,okpd2_codes,participant_inn_kpp,is_winner,okpd_list
0,44fz,0345200019825000037_,47,2025-04-10,Поставка фотобарабанов (совместимых) для много...,NaN,True,3000000.0,7842453565_470301001,Фотобарабан 40IU40R || Фотобарабан 60IU60S,26.20.40.120,F04E2C1D20D9B61B7C99B48EBAE5C0D7,1,[26.20.40.120]
1,44fz,0345200019825000038_,47,2025-04-10,Поставка фотобарабанов и блоков проявки (ориги...,NaN,True,3000000.0,7842453565_470301001,Блок проявки DV-1150 || Фотобарабан DK-1150 ||...,26.20.40.120,9F89EA463592C89553736DDFB9CBF73C,1,[26.20.40.120]
2,44fz,0345200019825000046_,47,2025-05-05,Поставка IP-видеорегистраторов и камер видеона...,NaN,True,1652750.0,7842453565_470301001,Камера видеонаблюдения,26.70.13.000 | 26.40.33.190,CB37D3E419F63C7D0833A6C6AE828793,1,"[26.70.13.000, 26.40.33.190]"
3,44fz,0345200019825000050_,47,2025-05-07,Поставка штемпельной продукции для филиала «Му...,NaN,True,141150.0,7842453565_470301001,Штемпель,NaN,58D8A3BA3536D51D39525FD1F8E8EEFC,1,[]
4,44fz,0345200019825000051_,47,2025-05-07,Поставка маршрутизаторов,NaN,True,1005000.0,7842453565_470301001,Маршрутизатор,NaN,1C8A8B3933EE4C539BC98F9EEC0C3C49,1,[]


In [18]:
def build_text(row):
    """Склеивает названия закупки, лота и товара в один текст.
    Берет одну строку таблицы, идет по трем текстовым колонкам,
    пропускает пустые, складывает непустые в корзину,
    склеивает через пробел и приводит к нижнему регистру.
    Прогоняет эту функцию по всем строкам и кладет результат в колонку text.
    """
    
    parts = []
    for col in ["purchase_name", "lot_name", "item_descriptions"]:
        v = row.get(col)
        if pd.notna(v) and str(v).strip():
            parts.append(str(v))
    return " ".join(parts).lower()

df["text"] = df.apply(build_text, axis=1)

In [19]:
display(df.head())

,fz,pn_lot,region_code,min_publish_date,purchase_name,lot_name,forsmallbiz,lot_price,customer_inn_kpp,item_descriptions,okpd2_codes,participant_inn_kpp,is_winner,okpd_list,text
0,44fz,0345200019825000037_,47,2025-04-10,Поставка фотобарабанов (совместимых) для много...,NaN,True,3000000.0,7842453565_470301001,Фотобарабан 40IU40R || Фотобарабан 60IU60S,26.20.40.120,F04E2C1D20D9B61B7C99B48EBAE5C0D7,1,[26.20.40.120],поставка фотобарабанов (совместимых) для много...
1,44fz,0345200019825000038_,47,2025-04-10,Поставка фотобарабанов и блоков проявки (ориги...,NaN,True,3000000.0,7842453565_470301001,Блок проявки DV-1150 || Фотобарабан DK-1150 ||...,26.20.40.120,9F89EA463592C89553736DDFB9CBF73C,1,[26.20.40.120],поставка фотобарабанов и блоков проявки (ориги...
2,44fz,0345200019825000046_,47,2025-05-05,Поставка IP-видеорегистраторов и камер видеона...,NaN,True,1652750.0,7842453565_470301001,Камера видеонаблюдения,26.70.13.000 | 26.40.33.190,CB37D3E419F63C7D0833A6C6AE828793,1,"[26.70.13.000, 26.40.33.190]",поставка ip-видеорегистраторов и камер видеона...
3,44fz,0345200019825000050_,47,2025-05-07,Поставка штемпельной продукции для филиала «Му...,NaN,True,141150.0,7842453565_470301001,Штемпель,NaN,58D8A3BA3536D51D39525FD1F8E8EEFC,1,[],поставка штемпельной продукции для филиала «му...
4,44fz,0345200019825000051_,47,2025-05-07,Поставка маршрутизаторов,NaN,True,1005000.0,7842453565_470301001,Маршрутизатор,NaN,1C8A8B3933EE4C539BC98F9EEC0C3C49,1,[],поставка маршрутизаторов маршрутизатор


## 3. Составление профилей поставщиков

Для каждого поставщика формируется профиль из его истории:
 - в каких ОКПД2 работал,
 - в каком регионе в основном работал (самый частый, т.е. мода),
 - диапазон сумм (10-й и 90-й перцентиль — отсекаем выбросы),
 - заказчиков, с которыми работал,
 - по каким ФЗ участвовал,
 - работает ли с малым бизнесом.

В дальнейшем профиль поставщика, запрашивающего рекомендации будет сравниваться с этими профилями.

In [22]:
def build_profiles(df):
    """Профиль каждого поставщика по его истории.
    
    Аргументы:
     df - датасет с лотами.

    Возвращаемое значение:
    профиль поставщика, содержащий сведения о его работе
    в прошлом: ОКПД, основной регион, диапазон цен,
    список заказачиков, ФЗ участия, опыт с малым бизнесом.
    """

    group = df.groupby("participant_inn_kpp")
    profiles = pd.DataFrame({
        
        # все ОКПД2, в которых участвовал
        "okpd": group["okpd_list"].apply(lambda s: set(x for lst in s for x in lst)),
       
        # самый частый регион
        "region": group["region_code"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None),
        
        # 10-й и 90-й перцентиль убирает выбросы
        "price_min": group["lot_price"].quantile(0.10),
        "price_max": group["lot_price"].quantile(0.90),
        
        # заказчики
        "customers": group["customer_inn_kpp"].apply(set),
        
        # ФЗ, по которым участвовал
        "fz": group["fz"].apply(set),
        
        # работал ли с малым бизнесом
        "smallbiz": group["forsmallbiz"].apply(lambda s: s.astype(str).str.lower().eq("true").any()),
    })
    return profiles


profiles = build_profiles(df)


In [23]:
print("\nПример профиля поставщика:")
print(profiles.iloc[12]) 


Пример профиля поставщика:
okpd                 {71.12.14.100}
region                           23
price_min                 7637620.2
price_max                10557810.6
customers    {2308077190_230801001}
fz                           {44fz}
smallbiz                       True
Name: 0009213065230FB100AD6DC3DC1E25D3, dtype: object


## 4. Скоринг лота

Оценка того, насколько лот подходит поставщику: чем больше число - тем выше лот в рекомендациях.
Профиль поставщика и данные лота и сравниваются по признакам датасета, присваиваются очки сходства и далее они суммируются.

Признаки и веса по умолчанию:
 - ОКПД2: 0,5
 - регион: 0,1,
 - сумма лота: 0,1,
 - уже работал с этим заказчиком: 0,2,
 - ФЗ лота (0,1).


In [26]:
def score_lot(supplier, lot, weights=None):
    """Насколько лот подходит поставщику -
    определяется как сумма очков.
    
    Аргументы:
     supplier — профиль поставщика (одна строка из profiles),
     lot — один лот (одна строка из df),
     weights=None — веса (необязательный, по умолчанию None,
     чтобы можно было калибровать).

    Возвращаемое значение:
    очки совпадения профиля поставщика с лотом.
    
    """
    
    w = weights or {"okpd": 0.5, "region": 0.1, "price": 0.1,
                    "customer": 0.2, "fz": 0.1}
    score = 0.0

    # конвертация в множество, т.к. с ним можно делать inner join
    lot_okpd = set(lot["okpd_list"])

    # учитывает нормализацию, чтобы лот с 10 кодами,
    # из которых совпал 1, не получал столько же,
    # сколько лот с 1 кодом, который совпал. 
    if lot_okpd and supplier["okpd"]:                                    # ← исправлено
        score += w["okpd"] * len(supplier["okpd"] & lot_okpd) / len(lot_okpd)

    # поставщик уже работал в регионе лота
    if lot["region_code"] == supplier["region"]:
        score += w["region"]
    
    # цена лота не пустая и попадает в диапазон поставщика
    if pd.notna(lot["lot_price"]) and supplier["price_min"] <= lot["lot_price"] <= supplier["price_max"]:
        score += w["price"]

    # поставщик уже работал с заказчиком лота
    if lot["customer_inn_kpp"] in supplier["customers"]:
        score += w["customer"]

    # поставщик уже работал с ФЗ лота   
    if lot["fz"] in supplier["fz"]:
        score += w["fz"]

    return score

## 5. Топ-10 рекомендуемых лотов для поставщика

Берем поставщика, выкидываем лоты, в которых он уже участвовал, и считаем очки по всем оставшимся лотам, сортируем, берём топ-10.

In [29]:
def recommend(sup_id, df, profiles, top_n=10, weights=None):
    """Находит топ-10 лотов по релевантности для поставщика.
    
    Аргументы:
     sup_id - идентификатор поставщика,
     df - датасет со статистикой лотов,
     profiles - таблица профилей поставщиков, 
     top_n - число рекомендаций, которые нужно вернуть,
     weights - веса для скоринга лотов (по умолчанию дефолтные,
     но можно калибровать)

    Возвращаемое значение:
    список лотов, рекомендованных поставщику для участия.
    """
    
    sup = profiles.loc[sup_id]

    # собирает лоты, в которых поставщик уже участвовал
    been = set(df.loc[df["participant_inn_kpp"] == sup_id, "pn_lot"])

    # собирает список лотов, в которых данный поставщик не участвовал 
    recomendations = df[~df["pn_lot"].isin(been)].drop_duplicates("pn_lot").copy()

    # оценка каждого лота
    recomendations["score"] = recomendations.apply(lambda lot: score_lot(sup, lot, weights), axis=1)
    
    return recomendations.sort_values("score", ascending=False).head(top_n)[
        ["pn_lot", "purchase_name", "region_code", "lot_price",
         "customer_inn_kpp", "okpd2_codes", "score"]
    ]

In [30]:
example_sup = profiles.index[0]
print("Поставщик:", example_sup)

rec = recommend(example_sup, df, profiles).copy()
rec["purchase_name"] = rec["purchase_name"].str.slice(0, 80) + "…"
rec["lot_price"] = (rec["lot_price"] / 1_000_000).round(1) 

display(rec.reset_index(drop=True))

Поставщик: 0001C57AD9F6440C1E6B67F0932126CF


,pn_lot,purchase_name,region_code,lot_price,customer_inn_kpp,okpd2_codes,score
0,0319100010325000012_,Оказание услуг и выполнение работ по содержани...,24,187.9,2460028834_246601001,42.11.20.300,1.0
1,0319100010325000014_,Выполнение работ по содержанию действующей сет...,24,121.3,2460028834_246601001,42.11.20.300,1.0
2,0319100010325000004_,Выполнение работ по содержанию действующей сет...,24,115.2,2460028834_246601001,42.11.20.300,1.0
3,0319100010325000005_,Выполнение работ по содержанию действующей сет...,24,160.1,2460028834_246601001,42.11.20.300,1.0
4,0319100010325000026_,Выполнение работ по содержанию искусственных с...,24,2.0,2460028834_246601001,42.11.20.300,0.9
5,0319100010325000006_,Оказание услуг и выполнение работ по содержан...,24,909.1,2460028834_246601001,42.11.20.300,0.9
6,0319100010325000008_,Оказание услуг и выполнение работ по содержани...,24,1051.4,2460028834_246601001,42.11.20.300,0.9
7,0319100010325000020_,Выполнение работ по установке элементов обустр...,24,26.6,2460028834_246601001,42.11.20.300,0.9
8,0319100010325000013_,Оказание услуг и выполнение работ по содержан...,24,1214.2,2460028834_246601001,42.11.20.300,0.9
9,0119300019825002345_,Выполнение работ по нанесению дорожной разметк...,24,200.0,2466215012_246301001,42.11.20.300,0.8


## 6. Проверка: история против рекомендаций

Проверка адекватности рекомендации: насколько опыт поставщика соотвествует рекомендациям.

In [33]:
def recommend_check(sup_id, df, profiles, top_n=10):
    hist = df.loc[df["participant_inn_kpp"] == sup_id,
                  ["okpd2_codes", "region_code", "lot_price", "is_winner"]].head(10).copy()
    hist["lot_price"] = (hist["lot_price"] / 1_000_000).round(1)

    rec = recommend(sup_id, df, profiles, top_n=top_n).copy()
    rec["lot_price"] = (rec["lot_price"] / 1_000_000).round(1)

    print(f"Поставщик: {sup_id}\n")
    
    print("=== ИСТОРИЯ УЧАСТИЯ ===")
    display(hist.rename(columns={"okpd2_codes": "ОКПД2",
                                 "lot_price": "млн",
                                 "region_code": "регион",
                                 "is_winner": "победа"
                                }))
    
    print("\n=== РЕКОМЕНДАЦИИ ===")
    display(rec[["okpd2_codes", "region_code", "lot_price", "score"]]
            .rename(columns={"okpd2_codes": "ОКПД2",
                             "lot_price": "млн",
                             "region_code": "регион",
                              "score": "балл"
                            }))


In [34]:
recommend_check(example_sup, df, profiles)

Поставщик: 0001C57AD9F6440C1E6B67F0932126CF

=== ИСТОРИЯ УЧАСТИЯ ===


,ОКПД2,регион,млн,победа
687002,42.11.20.300,24,2.5,1
834053,42.11.20.300,24,768.9,1



=== РЕКОМЕНДАЦИИ ===


,ОКПД2,регион,млн,балл
834058,42.11.20.300,24,187.9,1.0
834060,42.11.20.300,24,121.3,1.0
834050,42.11.20.300,24,115.2,1.0
834051,42.11.20.300,24,160.1,1.0
834070,42.11.20.300,24,2.0,0.9
834052,42.11.20.300,24,909.1,0.9
834054,42.11.20.300,24,1051.4,0.9
834066,42.11.20.300,24,26.6,0.9
834059,42.11.20.300,24,1214.2,0.9
698361,42.11.20.300,24,200.0,0.8
